# EXPRESSION DATA

In [8]:
import pandas as pd
from functools import reduce

# =========================================== #
# FILTER FOR 1:1 ORTHOLOGS AND RENAME COLUMNS #
# =========================================== #

kinases_orthologs = pd.read_csv("C:/Users/vicfo/Desktop/TFM/project/data/data_gathering/new/human_mouse_orthoKinases.txt", delimiter="\t")

# Filter out "ortholog_one2many"
kinases_1to1orthologs = kinases_orthologs[kinases_orthologs["Mouse homology type"] == "ortholog_one2one"].copy()

# Change column names: Gene stable ID version		Mouse homology type	Mouse gene name	Gene name
kinases_1to1orthologs.rename(columns={"Mouse homology type": "homology",
                                      "Mouse gene stable ID": "gene_id_mouse",
                                      "Mouse gene name": "gene_name_mouse",
                                      "Gene name": "gene_name_human",
                                      "Gene stable ID version": "gene_id_human"
                                      }, inplace=True)

kinases_1to1orthologs = kinases_1to1orthologs[["gene_id_human", "gene_name_human", "gene_id_mouse", "gene_name_mouse", "homology"]].copy()

# delete version of gene_id_human
kinases_1to1orthologs["gene_id_human"] = kinases_1to1orthologs["gene_id_human"].str.split(".").str[0]

# capitalize gene_name_mouse
kinases_1to1orthologs["gene_name_mouse"] = kinases_1to1orthologs["gene_name_mouse"].str.upper()

# Save it
kinases_1to1orthologs.to_csv("C:/Users/vicfo/Desktop/TFM/project/data/data_gathering/new/human_mouse_orthoKinases_one2one.txt", sep="\t", index=False)


# ================================================================================= #
# JOIN INFORMATION OF DIFFERENT TISSUES OF EACH SPECIE AND EXTRACT TPMs AND GENE ID #
# ================================================================================= #

# Data
datasets_mouse = {
    "ENCFF890GTZ": "adrenal_gland",
    "ENCFF707MBR": "brain",
    "ENCFF546IKJ": "muscle",
    "ENCFF519HZD": "heart",
    "ENCFF161ZNN": "liver"
}

datasets_human = {
    "ENCFF783MER": "adrenal_gland",
    "ENCFF725USM": "brain",
    "ENCFF296APA": "muscle",
    "ENCFF717ZEJ": "heart",
    "ENCFF658ESI": "liver",
}

def process_datasets(datasets_dict, species, kinases_1to1orthologs):

    # Empty dictionary to store processed datasets
    processed_datasets = {}
    
    # Access to every entry of the dictionary
    for file_name, tissue in datasets_dict.items():

        # Retrieve only gene name and expression column (TPM)
        file_path = f"C:/Users/vicfo/Desktop/TFM/project/Data/data_gathering/new/{file_name}.tsv"
        df = pd.read_csv(file_path, delimiter="\t")
        df = df[["gene_id", "TPM"]]
        df.rename(columns={"TPM": f"{tissue}_tpm_{species}"}, inplace=True)

        # Delete gene version
        df[f"gene_id_{species}"] = df["gene_id"].str.split(".").str[0]
        df.drop(columns=["gene_id"], inplace=True)

        print("Processing tissue:", tissue, "for species:", species)

        # Keep only entries that are in the kinase orthologs list
        df_filtered = pd.merge(df,kinases_1to1orthologs[[f"gene_id_{species}", f"gene_name_{species}"]],on=f"gene_id_{species}",how="inner",)

        # Store in "processed_datasets" dictionary
        processed_datasets[tissue] = df_filtered

    return processed_datasets

processed_datasets_mouse = process_datasets(datasets_mouse, "mouse", kinases_1to1orthologs)
processed_datasets_human = process_datasets(datasets_human, "human", kinases_1to1orthologs)


# ===================================== #
# JOIN DATASETS - MERGE HUMAN AND MOUSE #
# ===================================== #

def join_datasets(dictionary, species):

    # Merge different tissue expression dataframes into a single dataframe
    merged_df = reduce(
    lambda left, right: pd.merge(
        left, right, on=[f"gene_id_{species}", f"gene_name_{species}"], how="outer"
    ),
    list(dictionary.values()),
    )

    # Sort column names "gene_id_{species}", "gene_name_{species}" first, then the rest
    cols = merged_df.columns.tolist()
    cols.insert(0, cols.pop(cols.index(f"gene_id_{species}")))
    cols.insert(1, cols.pop(cols.index(f"gene_name_{species}")))
    merged_df = merged_df[cols]

    return merged_df

table_mouse = join_datasets(processed_datasets_mouse, "mouse")
table_human = join_datasets(processed_datasets_human, "human")

# Merge the result with mouse expression to create the final paired dataset
table_human = table_human.rename(columns={"gene_name_human": "gene_name"})
table_mouse = table_mouse.rename(columns={"gene_name_mouse": "gene_name"})

final_df = pd.merge(table_human, table_mouse, on="gene_name", how="inner")

# Sort column names "gene_id_human", "gene_name", "gene_id_mouse" first, then the rest
cols = final_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("gene_id_human")))
cols.insert(1, cols.pop(cols.index("gene_id_mouse")))
cols.insert(2, cols.pop(cols.index("gene_name")))
final_df = final_df[cols]

final_df.to_csv("C:/Users/vicfo/Desktop/TFM/project/data/data_gathering/new/merged_expression_humanmouse_kinases_1to1orthologs.txt", sep="\t", index=False)

Processing tissue: adrenal_gland for species: mouse
Processing tissue: brain for species: mouse
Processing tissue: muscle for species: mouse
Processing tissue: heart for species: mouse
Processing tissue: liver for species: mouse
Processing tissue: adrenal_gland for species: human
Processing tissue: brain for species: human
Processing tissue: muscle for species: human
Processing tissue: heart for species: human
Processing tissue: liver for species: human


# Filtering for Noise Reduction

##### (<1TPM cannot be distinguished from background noise)

In [10]:
genes = pd.read_csv("C:/Users/vicfo/Desktop/TFM/project/data/data_gathering/new/merged_expression_humanmouse_kinases_1to1orthologs.txt", sep="\t")

# ======= #
# < 1 TPM #
# ======= #

# Filter out genes with TPM < 1 in all tissues for both species
tissue_columns_human = [col for col in genes.columns if col.endswith("_tpm_human")]
tissue_columns_mouse = [col for col in genes.columns if col.endswith("_tpm_mouse")]
filtered_genes = genes[(genes[tissue_columns_human].ge(1).any(axis=1)) & (genes[tissue_columns_mouse].ge(1).any(axis=1))]

In [12]:
# Save 
filtered_genes.to_csv("C:/Users/vicfo/Desktop/TFM/project/data/data_gathering/new/new_expression_data.txt", sep="\t", index=False)

# Retrieve Promoters from Ensembl

In [13]:
import requests
import json
import time
import pandas as pd

In [14]:
def get_promoter_sequences_batch(gene_list, upstream=500):
    """
    Fetches promoter sequences in batches of 50 using POST requests.
    """
    url = "https://rest.ensembl.org/sequence/id"
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json"
    }
    
    results = {}
    
    # Ensembl POST limit is 50 IDs per request
    for i in range(0, len(gene_list), 50):
        print(f"Processing batch {i//50 + 1}")
        batch = list(gene_list[i:i+50])
        data = {
            "ids": batch,
            "type": "genomic",
            "expand_5prime": upstream
        }
        
        # Convert dictionary to JSON string for the POST body
        response = requests.post(url, headers=headers, data=json.dumps(data))
        
        if response.ok:
            batch_results = response.json()
            for entry in batch_results:
                # The prompt asks for ONLY the first 500bp (the expanded region)
                full_seq = entry['seq']
                # Ensembl returns expanded_upstream + gene_sequence. 
                # We slice the first 'upstream' characters to get just the promoter.
                results[entry['id']] = full_seq[:upstream]
        else:
            print(f"Error fetching batch starting at {i}: {response.status_code}")
            # Wait a second if we hit a rate limit
            if response.status_code == 429:
                time.sleep(1)
        
    return results

In [15]:
expression_data = pd.read_csv("C:/Users/vicfo/Desktop/TFM/project/data/data_gathering/new/new_expression_data.txt", sep="\t")
human_ids = expression_data["gene_id_human"].tolist()
mouse_ids = expression_data["gene_id_mouse"].tolist()

# Retrieve in batches
promoters_human = get_promoter_sequences_batch(human_ids)
promoters_mouse = get_promoter_sequences_batch(mouse_ids)

# --- Save to FASTA ---
def save_fasta(data_dict, filename):
    with open(filename, "w") as f:
        for gene, seq in data_dict.items():
            if seq: # Ensure sequence exists
                f.write(f">{gene}\n{seq}\n")

human_file = f"C:/Users/vicfo/Desktop/TFM/project/data/data_gathering/new/promoter_kinases_human_new.fasta"
mouse_file = f"C:/Users/vicfo/Desktop/TFM/project/data/data_gathering/new/promoter_kinases_mouse_new.fasta"

save_fasta(promoters_human, human_file)
save_fasta(promoters_mouse, mouse_file)

print("Done! Batch retrieval completed.")

Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processing batch 5
Processing batch 6
Processing batch 7
Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processing batch 5
Processing batch 6
Processing batch 7
Done! Batch retrieval completed.
